In [2]:
from Crypto.Cipher import AES
from Crypto.Random import get_random_bytes

BLOCK_SIZE = 16

In [3]:
def xor_bytes(a, b):
    return bytes(x ^ y for x, y in zip(a, b))

In [4]:
def pad(data):
    padding_len = BLOCK_SIZE - len(data) % BLOCK_SIZE
    return data + bytes([padding_len]) * padding_len

In [5]:
def unpad(data):
    padding_len = data[-1]
    return data[:-padding_len]

In [7]:
def aes_cbc_encrypt(key, plaintext, iv):
    cipher = AES.new(key, AES.MODE_ECB)
    plaintext = pad(plaintext)
    ciphertext = b''
    previous = iv

    for i in range(0, len(plaintext), BLOCK_SIZE):
        block = plaintext[i:i+BLOCK_SIZE]
        xored = xor_bytes(block, previous)
        encrypted = cipher.encrypt(xored)
        ciphertext += encrypted
        previous = encrypted

    return ciphertext


In [8]:
def aes_cbc_decrypt(key, ciphertext, iv):

    cipher = AES.new(key, AES.MODE_ECB)

    plaintext = b''

    previous = iv

    for i in range(0, len(ciphertext), BLOCK_SIZE):

        block = ciphertext[i:i+BLOCK_SIZE]

        decrypted = cipher.decrypt(block)

        plain_block = xor_bytes(
            decrypted,
            previous
        )

        plaintext += plain_block

        previous = block

    return unpad(plaintext)

In [10]:
def aes_ctr_encrypt(key, plaintext, nonce):
    cipher = AES.new(key, AES.MODE_ECB)
    ciphertext = b''
    counter = 0

    for i in range(0, len(plaintext), BLOCK_SIZE):
        block = plaintext[i:i+BLOCK_SIZE]
        counter_block = (
            nonce +
            counter.to_bytes(8, 'big')
        )
        keystream = cipher.encrypt(
            counter_block
        )
        encrypted = xor_bytes(
            block,
            keystream[:len(block)]
        )
        ciphertext += encrypted
        counter += 1

    return ciphertext

In [11]:
aes_ctr_decrypt = aes_ctr_encrypt

In [12]:
key = get_random_bytes(16)
iv = get_random_bytes(16)

In [13]:
message = b"Implementing AES-CBC and AES-CTR from scratch using only raw AES block operations"

In [14]:
cbc_cipher = aes_cbc_encrypt(
    key,
    message,
    iv
)

In [15]:
cbc_plain = aes_cbc_decrypt(
    key,
    cbc_cipher,
    iv
)

In [16]:
print("CBC Cipher:", cbc_cipher.hex())
print("CBC Plain:", cbc_plain)

CBC Cipher: f3702b52c3bf083fc9526780a98a1976d7ab5517797a89cd356572b9810571607f3d2ba744017d337dfaa9947f38abb0752be52e6f9716fba449fecca5491bba9e8b1cf35fa6efb6dbde671f15f4b6108fb80e40e1fe755658d5d87ce2959a5e
CBC Plain: b'Implementing AES-CBC and AES-CTR from scratch using only raw AES block operations'


In [17]:
nonce = get_random_bytes(8)

In [18]:
ctr_cipher = aes_ctr_encrypt(
    key,
    message,
    nonce
)

In [19]:
ctr_plain = aes_ctr_encrypt(
    key,
    ctr_cipher,
    nonce
)

In [20]:
print("\nCTR Cipher:",
      ctr_cipher.hex())

print("CTR Plain:",
      ctr_plain)


CTR Cipher: ff39c2ac876b2eb11c403e50530c6a9322bb8cb0899459b2d1095f3ef4e653bda6450712068cfe0cbbdefe264fd17ce5e7654c41978fd4764c8f219fd6668552983be24e090c9078008f02821f0c922ca4
CTR Plain: b'Implementing AES-CBC and AES-CTR from scratch using only raw AES block operations'
